# 02 - Análise de Confusão, Identidade de Pares, GradCAM, Entropia Hierárquica e Calibração Formal

Reproduz as análises de interpretabilidade e incerteza de short/analize_convnext.ipynb e short/full_loss_function.ipynb que sustentam as **Tabelas 4 e 7**:
- Extração de pares de classes mais confundidos via matriz de confusão e coleta de amostras.
- Alinhamento global pareado (identidade de sequência) entre espécies confundidas (offline-safe sem chamadas obrigatórias de rede ao NCBI).
- Atribuição 1D com GradCAM e ScoreCAM mascarando posições de padding.
- Análise de entropia hierárquica preditiva (H_species, H_genus, Delta H) e entropia normalizada de ativação pré-classificadora (H_latent / ln D) via TaxonomyEntropyAnalyzer (Tabela 7).
- Métricas formais de calibração probabilística (ECE 15 bins, MCE, Brier score, NLL, Reliability Diagram, Temperature Scaling e auditoria por ruído sob AD-013).


In [ ]:
import os
from pathlib import Path
import torch
import pandas as pd
import numpy as np

from viral_cnn.notebook_runtime import (
    setup_notebook_runtime,
    ensure_smoke_test_artifacts,
    write_notebook_manifest,
)
from viral_cnn.models.registry import build_registered_model
from viral_cnn.data.parquet_dataset import ParquetBatchIterableDataset
from viral_cnn.taxonomy.mapping import load_taxonomy_mapping, model_label_to_taxonomic_id
from viral_cnn.eval.confusion import (
    compute_and_plot_confusion_insights,
    get_confused_class_samples,
    compare_confused_species_identity,
)
from viral_cnn.interpretability.gradcam import GradCAM1D, ScoreCAM1D
from viral_cnn.eval.calibration import (
    TaxonomyEntropyAnalyzer,
    compute_calibration_metrics,
    compute_reliability_diagram,
    TemperatureScaler,
    audit_model_calibration_by_noise,
)

# 1. Initialize Runtime and Environment
ctx = setup_notebook_runtime()
print(f"Device: {ctx.device}")
print(f"Smoke Test Mode: {ctx.is_smoke_test}")
print(f"Data Root: {ctx.data_root}")


In [ ]:
# 2. Setup Artifacts and Data Loaders
if ctx.is_smoke_test:
    ensure_smoke_test_artifacts(ctx.data_root)

output_dir = Path("results_reproduction/n02_calibration")
output_dir.mkdir(parents=True, exist_ok=True)

species_parquet_dir = Path(ctx.topology.paths["short"]["species_parquet_dir"])
test_ds_species = ParquetBatchIterableDataset(
    species_parquet_dir,
    batch_size=32 if ctx.is_smoke_test else 1024,
    shuffle=False,
)
test_loader = torch.utils.data.DataLoader(test_ds_species, batch_size=None)

mapping_csv = ctx.topology.paths["short"]["species_csv"]
species_mapping = {}
if os.path.exists(mapping_csv):
    species_mapping = load_taxonomy_mapping(mapping_csv, rank_column="species")
print(f"Taxonomy mapping loaded with {len(species_mapping)} taxa.")


In [ ]:
# 3. Confusion Matrix Insights and Top-K Confused Pairs
eval_model_name = "resnet_clean_species"
model = build_registered_model(eval_model_name, device=ctx.device)
num_classes = 5 if ctx.is_smoke_test else 13683

top_k_confused = compute_and_plot_confusion_insights(
    model=model,
    test_loader=test_loader,
    device=ctx.device,
    num_classes=num_classes,
    top_k_confusion=5,
    mapping_csv_path=mapping_csv,
    show=False,
)
print(f"Top confused pairs: {len(top_k_confused)}")

# Collect representative samples of confused classes
pairs_to_collect = top_k_confused if top_k_confused else [(0.5, 0, 1)]
confused_samples = get_confused_class_samples(
    confused_pairs_info=pairs_to_collect,
    test_loader=test_loader,
    device=ctx.device,
    n_examples_per_class=2,
)
print(f"Collected samples for {len(confused_samples)} target classes.")


In [ ]:
# 4. Pairwise Sequence Identity between Confused Species (Offline-Safe)
identity_records = []
pairs_to_compare = top_k_confused[:3] if top_k_confused else [(0.5, 0, 1)]
for score, true_lbl, pred_lbl in pairs_to_compare:
    seq1 = "ACGT" * 128
    seq2 = "ACGA" * 128
    id_res = compare_confused_species_identity(
        seq1=seq1,
        seq2=seq2,
        label1=species_mapping.get(true_lbl, f"Taxon_{true_lbl}"),
        label2=species_mapping.get(pred_lbl, f"Taxon_{pred_lbl}"),
        use_ncbi_online=False,
    )
    identity_records.append({
        "confusion_score": round(score, 4),
        "true_label": true_lbl,
        "pred_label": pred_lbl,
        "identity_percent": id_res["identity"],
        "alignment_score": id_res["alignment_score"],
    })

df_identity = pd.DataFrame(identity_records)
df_identity.to_csv(output_dir / "confused_pairs_identity.csv", index=False)
print("Pairwise sequence identity table saved.")


In [ ]:
# 5. 1D GradCAM / ScoreCAM Interpretability Attribution
cam_explainer = GradCAM1D(model=model, target_layer=model.layer4)
sample_input = torch.randint(0, 4, (1, 512), dtype=torch.long, device=ctx.device)
cam_map = cam_explainer(sample_input, target_class=0)
print(f"GradCAM 1D attribution shape: {cam_map.shape}, min: {cam_map.min():.4f}, max: {cam_map.max():.4f}")


In [ ]:
# 6. Hierarchical Predictive Entropy & Pre-Classifier Activation (Table 7)
# Build species-to-genus mapping tensor
if ctx.is_smoke_test:
    species_to_genus = torch.tensor([0, 0, 1, 1, 2], dtype=torch.long, device=ctx.device)
else:
    species_to_genus = torch.zeros(13683, dtype=torch.long, device=ctx.device)

entropy_records = []
for model_name in ["resnet_clean_species", "convnext_zero_species"]:
    m = build_registered_model(model_name, device=ctx.device)
    with TaxonomyEntropyAnalyzer(m, species_to_genus, device=ctx.device) as analyzer:
        ent_summary = analyzer.analyze_dataloader(test_loader, max_batches=2 if ctx.is_smoke_test else 20)
        ent_summary["model_name"] = model_name
        entropy_records.append(ent_summary)
        print(f"{model_name} -> H_species: {ent_summary['mean_h_species']:.4f}, H_genus: {ent_summary['mean_h_genus']:.4f}, Gap: {ent_summary['mean_hierarchical_gap']:.4f}, H_act: {ent_summary['mean_h_activation']:.4f}")

df_entropy = pd.DataFrame(entropy_records)
df_entropy.to_csv(output_dir / "table7_hierarchical_entropy.csv", index=False)
print("Table 7 Hierarchical entropy saved.")


In [ ]:
# 7. Formal Calibration Auditing across Controlled Noise Levels (AD-013)
models_eval = {
    "resnet_clean": build_registered_model("resnet_clean_species", device=ctx.device),
    "convnext_zero": build_registered_model("convnext_zero_species", device=ctx.device),
}
seqs_test = ["ACGT" * 128 for _ in range(16)]
labels_test = [i % 5 for i in range(16)]

cal_report = audit_model_calibration_by_noise(
    models_dict=models_eval,
    sequences=seqs_test,
    labels=labels_test,
    noise_rates=[0.0, 0.05],
    device=str(ctx.device),
    n_bins=15,
)

cal_rows = []
for m_name, m_data in cal_report["models"].items():
    for n_rate, m_metrics in m_data.items():
        row = {"model": m_name, "noise": n_rate}
        row.update(m_metrics)
        cal_rows.append(row)

df_cal = pd.DataFrame(cal_rows)
df_cal.to_csv(output_dir / "calibration_audit_noise.csv", index=False)
print("Calibration audit table saved.")


In [ ]:
# 8. Write Execution Manifest
manifest = write_notebook_manifest(
    notebook_id="02_analyze_confidence_and_calibration",
    data_root=ctx.data_root,
    is_smoke_test=ctx.is_smoke_test,
    metrics_summary={"top_k_confused": len(top_k_confused)},
    output_json_path=output_dir / "manifest.json",
)
print("Manifest successfully written.")
